## Initialization Steps

- set up the transformers library
- get some nice tools for doing things with strings


In [ ]:
from transformers import pipeline
import string

# Initialize once
unmasker = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased"
)

# Simple stopword set (extend freely)
STOPWORDS = {
    "the", "a", "an", "of", "and", "to", "in", "on",
    "for", "with", "at", "by", "from", "as", "is"
}

def complete_mask(sentence, n=5, extra_stopwords=None):
    """
    Return top-n filtered BERT mask completions.
    Removes punctuation, WordPiece fragments, and stopwords.
    """

    if "[MASK]" not in sentence:
        raise ValueError("Input sentence must contain '[MASK]' token.")

    if extra_stopwords is None:
        extra_stopwords = set()

    stopwords = STOPWORDS.union(extra_stopwords)

    raw_results = unmasker(sentence, top_k=4*n)

    filtered = []

    for r in raw_results:
        token = r["token_str"].strip().lower()

        # remove subword fragments
        if token.startswith("##"):
            continue

        # remove punctuation-only tokens
        if all(c in string.punctuation for c in token):
            continue

        # remove stopwords
        if token in stopwords:
            continue

        # keep alphabetic tokens only
        if not token.isalpha():
            continue

        filtered.append({
            "completion": token,
            "score": r["score"]
        })

        if len(filtered) == n:
            break

    return filtered

In [ ]:
examples = [
    "Hamilton College is located in [MASK], New York.",
    "Linear algebra is one of the most [MASK] subjects in mathematics.",
    "The theorem follows immediately from the [MASK].",
    "The man went to work. He is a [MASK].",
    "The woman went to work. She is a [MASK]."
]

In [ ]:
for sentence in examples:
    output = complete_mask(sentence, 10)
    for o in output:
        print(f"{sentence} {o['completion']:>12} score={o['score']:.4f}")

In [ ]:
complete_mask("[MASK] is the best animal.",5) #try adding "A" to the front

In [ ]:
complete_mask("The Dalai Lama likes [MASK] philosophy.",10)